# Lab 04.3 — HSV and Color Masks

## Objectives

- understand why HSV is useful for color segmentation;
- create binary masks with `cv2.inRange()`;
- compare masks for several colors;
- clean a mask using morphological operations.

The experiment first uses the controlled image from Lab 04.1 and then a webcam frame.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image = cv2.imread("images/colored_shapes.png")
if image is None:
    raise FileNotFoundError("Run Lab04.1 once to generate images/colored_shapes.png")

hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

## 1. HSV thresholds

OpenCV uses Hue values from 0 to 179. Red wraps around the ends of this interval, therefore red usually needs two Hue ranges.

In [ ]:
ranges = {
    "blue":   ((100, 100, 80), (130, 255, 255)),
    "green":  ((35, 80, 80),   (85, 255, 255)),
    "yellow": ((20, 100, 100), (35, 255, 255))
}

masks = {}

for name, (lower, upper) in ranges.items():
    masks[name] = cv2.inRange(hsv, np.array(lower), np.array(upper))

red1 = cv2.inRange(hsv, np.array((0, 100, 100)), np.array((10, 255, 255)))
red2 = cv2.inRange(hsv, np.array((170, 100, 100)), np.array((179, 255, 255)))
masks["red"] = cv2.bitwise_or(red1, red2)

In [ ]:
for name, mask in masks.items():
    plt.figure(figsize=(8, 3))
    plt.imshow(mask, cmap="gray")
    plt.title(name.upper() + " mask")
    plt.axis("off");

## 2. Apply a mask to the original image

In [ ]:
green_result = cv2.bitwise_and(image, image, mask=masks["green"])
green_rgb = cv2.cvtColor(green_result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 4))
plt.imshow(green_rgb)
plt.title("Pixels selected by the green mask")
plt.axis("off");

## 3. Morphological cleanup

Real webcam images contain noise. Opening removes small isolated regions; closing can fill small holes.

In [ ]:
kernel = np.ones((5, 5), np.uint8)

clean = cv2.morphologyEx(masks["green"], cv2.MORPH_OPEN, kernel)
clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel)

plt.figure(figsize=(8, 3))
plt.imshow(clean, cmap="gray")
plt.title("Cleaned mask")
plt.axis("off");

## 4. Try the same operation on one webcam frame

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

ret, frame = cap.read()
cap.release()

if ret:
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv_frame, np.array((35, 80, 50)), np.array((85, 255, 255)))
    
    plt.figure(figsize=(10, 5))
    plt.imshow(mask, cmap="gray")
    plt.title("Green mask from webcam")
    plt.axis("off")
else:
    print("No frame captured. Check the webcam as in Lab04.2.")

## Exercise

Use a strongly colored object in front of the camera.

1. Start with green.
2. Adjust the HSV limits until the object is selected reliably.
3. Observe what happens when lighting changes.
4. Repeat for one additional color.